Ramalah Amir </br>
23i-2644 </br>
DS-6C </br>


# Loading the Dataset


In [ ]:
import gzip
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import os

random.seed(42)
torch.manual_seed(42)


def load_data(file_path, max_samples=15000):
    data = []

    with gzip.open(file_path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            record = json.loads(line)

            if "reviewText" in record and "overall" in record:
                data.append(
                    {"review": record["reviewText"], "rating": record["overall"]}
                )
    return data


cat1 = load_data("Arts_Crafts_and_Sewing_5.json.gz", 15000)
cat2 = load_data("Digital_Music_5.json.gz", 15000)
cat3 = load_data("Movies_and_TV_5.json.gz", 15000)

all_data = cat1 + cat2 + cat3
random.shuffle(all_data)
print(f"Total samples loaded: {len(all_data)}")

Total samples loaded: 44980


# Extracting Targets


In [ ]:
def process_samples(data):
    processed = []
    for d in data:
        text = str(d["review"]).lower()
        rating = d["rating"]

        # Target 01: sentiment classification
        # 1-2 -> Negative (0), 3 -> Neutral (1), 4-5 -> Positive (2)
        if rating <= 2:
            sentiment = 0
        elif rating == 3:
            sentiment = 1
        else:
            sentiment = 2

        # Target 02: Length Classification
        length = len(text.split())
        if length < 20:
            length_class = 0  # Short
        elif length <= 50:
            length_class = 1  # Medium
        else:
            length_class = 2  # Long

        processed.append(
            {"text": text, "sentiment": sentiment, "length_class": length_class}
        )
    return processed


processed_data = process_samples(all_data)

# Split 70% Train, 15% Val, 15% Test
n = len(processed_data)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

train_data = processed_data[:train_end]
val_data = processed_data[train_end:val_end]
test_data = processed_data[val_end:]

print(f"Train samples: {len(train_data)}")
print(f"Val samples:   {len(val_data)}")
print(f"Test samples:  {len(test_data)}")

# Tokenization and Vocabulary Building


In [ ]:
def simple_tokenize(text):
    text = re.sub(r"([^a-z])", r" \1 ", text)
    return text.split()


# Build vocabulary ONLY from the training set to prevent data leakage
vocab_counter = Counter()
for d in train_data:
    tokens = simple_tokenize(d["text"])
    vocab_counter.update(tokens)

# Selecting top frequent words
vocab_size = 10000

special_tokens = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
most_common = [word for word, count in vocab_counter.most_common(vocab_size - 4)]
vocab_tokens = special_tokens + most_common

word2idx = {w: i for i, w in enumerate(vocab_tokens)}
idx2word = {i: w for i, w in enumerate(vocab_tokens)}


def text_to_indices(text, max_len=100):
    tokens = simple_tokenize(text)
    indices = [word2idx.get(t, word2idx["<UNK>"]) for t in tokens]

    # Truncation
    if len(indices) > max_len:
        indices = indices[:max_len]
    # Padding
    if len(indices) < max_len:
        indices += [word2idx["<PAD>"]] * (max_len - len(indices))

    return indices

# Creating DataLoaders


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, data, max_len=100):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        indices = text_to_indices(item["text"], self.max_len)
        return {
            "input_ids": torch.tensor(indices, dtype=torch.long),
            "sentiment": torch.tensor(item["sentiment"], dtype=torch.long),
            "length_class": torch.tensor(item["length_class"], dtype=torch.long),
            "raw_text": item["text"],
        }


BATCH_SIZE = 64
MAX_LEN = 100

train_dataset = ReviewDataset(train_data, max_len=MAX_LEN)
val_dataset = ReviewDataset(val_data, max_len=MAX_LEN)
test_dataset = ReviewDataset(test_data, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Encoder Transformer


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert (
            d_model % num_heads == 0
        ), "d_model must be cleanly divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear layers for Query, Key, Value
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        # 1. Linear projections
        # Reshape to (batch_size, num_heads, seq_len, d_k)
        Q = self.W_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 2. Scaled Dot-Product Attention: (Q * K^T) / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            # Apply mask (fill padding with very large negative number before softmax)
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)

        # 3. Multiply by Values
        context = torch.matmul(attn, V)

        # 4. Concatenate heads back together
        context = (
            context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        )

        # 5. Final output projection
        return self.W_o(context)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-head attention with residual connection and layer norm
        attn_out = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed-forward network with residual connection and layer norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # Using sine/cosine frequencies
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)  # Add batch dimension

    def forward(self, x):
        # Add positional encodings to the input embeddings
        return x + self.pe[:, : x.size(1), :].to(x.device)


class EncoderTransformerModel(nn.Module):
    """Full Encoder framework combining all parts to perform classification tasks."""

    def __init__(self, vocab_size, d_model=64, num_heads=4, d_ff=128, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)  # 0 is PAD ID
        self.pos_encoder = PositionalEncoding(d_model)

        self.layers = nn.ModuleList(
            [
                TransformerEncoderLayer(d_model, num_heads, d_ff)
                for _ in range(num_layers)
            ]
        )

        # Classification Heads
        self.sentiment_head = nn.Linear(d_model, 3)
        self.length_head = nn.Linear(d_model, 3)

    def forward(self, input_ids):
        # Create mask for padding tokens
        # Shape matches attention matrix dimensions: (batch_size, 1, 1, seq_len)
        mask = (input_ids != 0).unsqueeze(1).unsqueeze(2)

        x = self.embedding(input_ids)
        x = self.pos_encoder(x)

        # Pass through all transformer layers
        for layer in self.layers:
            x = layer(x, mask)

        # Creating a single unified sentence embedding (Global Average Pooling ignoring PAING)
        # Sequence mask: (batch_size, seq_len, 1)
        seq_mask = (input_ids != 0).unsqueeze(-1).float()
        sum_embeddings = (x * seq_mask).sum(dim=1)
        valid_lengths = seq_mask.sum(dim=1).clamp(min=1)
        final_embedding = sum_embeddings / valid_lengths

        # Get Predictions
        sentiment_pred = self.sentiment_head(final_embedding)
        length_pred = self.length_head(final_embedding)

        return sentiment_pred, length_pred, final_embedding

# Training setup


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = EncoderTransformerModel(vocab_size=len(word2idx)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

epochs = 1

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct_sentiment = 0
    correct_length = 0
    total = 0

    for i, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        sentiment = batch["sentiment"].to(device)
        length_class = batch["length_class"].to(device)

        optimizer.zero_grad()
        sentiment_pred, length_pred, _ = model(input_ids)

        # Combined Loss calculation
        loss_sent = criterion(sentiment_pred, sentiment)
        loss_len = criterion(length_pred, length_class)
        loss = loss_sent + loss_len

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct_sentiment += (sentiment_pred.argmax(1) == sentiment).sum().item()
        correct_length += (length_pred.argmax(1) == length_class).sum().item()
        total += input_ids.size(0)

        if (i + 1) % 100 == 0:
            print(f"Batch {i+1} | Loss: {loss.item():.4f}")

    train_acc_sent = correct_sentiment / total
    train_acc_len = correct_length / total

    print(
        f"Epoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f} | Sent Acc: {train_acc_sent:.4f} | Len Acc: {train_acc_len:.4f}"
    )

# Extracting and Saving Training Embeddings


In [ ]:
model.eval()
train_embeddings = []
train_texts = []

with torch.no_grad():
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        _, _, emb = model(input_ids)
        train_embeddings.append(emb.cpu())
        train_texts.extend(batch["raw_text"])

train_embeddings = torch.cat(train_embeddings, dim=0)

# Save embeddings
torch.save(train_embeddings, "train_embeddings.pt")
print(f"Saved {train_embeddings.size(0)} embeddings to disk.")

# Retrieval Module


In [ ]:
# Pre-calculate normalized embeddings to make cosine similarity a simple dot product
norm_embeddings = F.normalize(train_embeddings, p=2, dim=1)


def retrieve_similar(query_embedding, k=3):
    # Normalize query
    query_norm = F.normalize(query_embedding, p=2, dim=0)

    # Compute similarity with all stored dataset items
    similarities = torch.matmul(norm_embeddings, query_norm)

    # Find top K values and indices
    top_scores, top_indices = torch.topk(similarities, k)

    results = []
    for score, idx in zip(top_scores, top_indices):
        results.append({"score": score.item(), "text": train_texts[idx]})
    return results


sample_emb = train_embeddings[0]
sample_text = train_texts[0]

print("Original Query Text:", sample_text)
print("=" * 40)
retrieved = retrieve_similar(sample_emb, k=3)
for i, res in enumerate(retrieved):
    print(f"Rank {i+1} | Score (Cosine Sim): {res['score']:.4f}")
    print(f"Retrieved Text: {res['text']}\n")

# Decoder Transformer


In [ ]:
target_texts = []
subset_size = min(1000, len(train_data))

for i in range(subset_size):
    text = train_data[i]["text"]
    sent = (
        "positive"
        if train_data[i]["sentiment"] == 2
        else "neutral" if train_data[i]["sentiment"] == 1 else "negative"
    )
    length = (
        "long"
        if train_data[i]["length_class"] == 2
        else "medium" if train_data[i]["length_class"] == 1 else "short"
    )

    sim_results = retrieve_similar(train_embeddings[i], k=2)
    sim_reviews = " ".join(
        [res["text"][:30] for res in sim_results]
    )  # using small snippet to keep context short

    # Structured string input formatting
    prompt = f"Review: {text[:30]} Sentiment: {sent} Feature: {length} Similar: {sim_reviews} Explanation: This is a {sent} review that is {length}."
    target_texts.append(prompt)


def text_to_indices_decoder(text, max_len=100):
    tokens = ["<SOS>"] + simple_tokenize(text) + ["<EOS>"]
    indices = [word2idx.get(t, word2idx["<UNK>"]) for t in tokens]
    if len(indices) > max_len:
        indices = indices[: max_len - 1] + [word2idx["<EOS>"]]
    if len(indices) < max_len:
        indices += [word2idx["<PAD>"]] * (max_len - len(indices))
    return indices


dec_indices = [text_to_indices_decoder(t) for t in target_texts]
dec_dataset = torch.tensor(dec_indices, dtype=torch.long)
dec_loader = DataLoader(dec_dataset, batch_size=32, shuffle=True)